# Notebook 03 — Factorization Machine (Baseline)

**Fields used:** `user_id`, `movie_id`, 19 binary genre fields, 30 binary tag fields  
**Split:** time-aware (last 20% of interactions by timestamp → test)  
**Output:** `models/fm_model.pt`, `models/fm_config.json`, `models/item_embeddings.npy`

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from src.fm_model import FactorizationMachine
from src.data_utils import load_ratings, load_movies, load_tags, build_id_mappings, train_test_split_by_time

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Load Data

In [ ]:
DATA_DIR = Path('../data/raw')

ratings = pd.read_csv(DATA_DIR / 'ratings.csv')
movies  = pd.read_csv(DATA_DIR / 'movies.csv')
tags    = pd.read_csv(DATA_DIR / 'tags.csv')

print(f'Ratings : {len(ratings):,} | Users: {ratings.userId.nunique()} | Movies: {ratings.movieId.nunique()}')
print(f'Movies  : {len(movies):,}')
print(f'Tags    : {len(tags):,} | Unique: {tags.tag.nunique()} | Tagged movies: {tags.movieId.nunique()}')

## 2. Genre Encoding (19 binary fields)

In [ ]:
ALL_GENRES = sorted(
    g for g in movies.genres.str.split('|').explode().unique()
    if g != '(no genres listed)'
)
print(f'{len(ALL_GENRES)} genres:', ALL_GENRES)

for g in ALL_GENRES:
    movies[g] = movies.genres.str.contains(g, regex=False).astype(int)

## 3. Tag Encoding (top-30 content tags, binary fields)

In [ ]:
# exclude meta-tags that describe user behaviour rather than movie content
META_TAGS = {'in netflix queue'}

content_tags = tags[~tags.tag.str.lower().isin(META_TAGS)]
COMMON_TAGS  = content_tags.tag.str.lower().value_counts().head(30).index.tolist()
print(f'{len(COMMON_TAGS)} tags selected:', COMMON_TAGS[:10], '...')

# aggregate tags per movie into a set of lower-cased tag strings
movie_tag_sets = (
    tags.groupby('movieId')['tag']
    .apply(lambda x: set(t.lower() for t in x))
    .to_dict()
)

# binary flag per tag per movie
for t in COMMON_TAGS:
    col = f'tag_{t}'
    movies[col] = movies['movieId'].map(
        lambda mid, _t=t: int(_t in movie_tag_sets.get(mid, set()))
    )

TAG_COLS = [f'tag_{t}' for t in COMMON_TAGS]
tagged = (movies[TAG_COLS].sum(axis=1) > 0).sum()
print(f'Movies with at least one top-30 tag: {tagged} / {len(movies)}')

## 4. Merge Features into Ratings + ID Mappings

In [ ]:
FEATURE_COLS = ALL_GENRES + TAG_COLS
ratings = ratings.merge(movies[['movieId'] + FEATURE_COLS], on='movieId', how='left')

user_mapping = build_id_mappings(ratings, 'userId')
item_mapping = build_id_mappings(ratings, 'movieId')

n_users = len(user_mapping)
n_items = len(item_mapping)
# 2 = cardinality of each binary field (0 or 1)
field_dims = [n_users, n_items] + [2] * len(FEATURE_COLS)

print(f'n_users={n_users} | n_items={n_items} | genre_fields={len(ALL_GENRES)} | tag_fields={len(TAG_COLS)}')
print(f'Total fields: {len(field_dims)} | Embedding table size: {sum(field_dims):,}')

train_df, test_df = train_test_split_by_time(ratings)
print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')

## 5. PyTorch Dataset

In [ ]:
class MovieLensDataset(Dataset):
    def __init__(self, df, user_mapping, item_mapping, feature_cols):
        self.users    = df['userId'].map(user_mapping).values
        self.items    = df['movieId'].map(item_mapping).values
        self.features = df[feature_cols].values.astype(np.int64)
        self.ratings  = df['rating'].values.astype(np.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        x = np.concatenate([[self.users[idx], self.items[idx]], self.features[idx]])
        return torch.tensor(x, dtype=torch.long), torch.tensor(self.ratings[idx])

BATCH = 512
train_ds = MovieLensDataset(train_df, user_mapping, item_mapping, FEATURE_COLS)
test_ds  = MovieLensDataset(test_df,  user_mapping, item_mapping, FEATURE_COLS)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH)
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

## 6. Train FM

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_n = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_loss += criterion(pred, y).item() * len(y)
            total_n    += len(y)
    return (total_loss / total_n) ** 0.5


def train_model(model, train_loader, test_loader, epochs=30, lr=1e-3, weight_decay=0, device=DEVICE):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    history = {'train_rmse': [], 'test_rmse': []}
    best_rmse, best_state = float('inf'), None

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
            total_n    += len(y)
        scheduler.step()

        train_rmse = (total_loss / total_n) ** 0.5
        test_rmse  = evaluate(model, test_loader, criterion, device)
        history['train_rmse'].append(train_rmse)
        history['test_rmse'].append(test_rmse)

        # early stopping: keep best test checkpoint
        if test_rmse < best_rmse:
            best_rmse  = test_rmse
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:02d}/{epochs} | train RMSE {train_rmse:.4f} | test RMSE {test_rmse:.4f}')

    # restore best checkpoint
    model.load_state_dict(best_state)
    print(f'Best test RMSE: {best_rmse:.4f}')
    return model, history


fm_model = FactorizationMachine(field_dims=field_dims, embed_dim=32)
print(f'Parameters: {sum(p.numel() for p in fm_model.parameters()):,}')
fm_model, fm_history = train_model(fm_model, train_loader, test_loader, epochs=30, weight_decay=0)

## 7. Loss Curves

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(fm_history['train_rmse'], label='Train RMSE')
plt.plot(fm_history['test_rmse'],  label='Test RMSE')
plt.xlabel('Epoch'); plt.ylabel('RMSE')
plt.title('FM — Training Curve (genres + tags features)')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Best test RMSE: {min(fm_history["test_rmse"]):.4f}')

## 8. Save Artifacts

In [ ]:
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

torch.save(fm_model.state_dict(), MODELS_DIR / 'fm_model.pt')

fm_config = {
    'model_type'   : 'fm',
    'field_dims'   : field_dims,
    'embed_dim'    : 32,
    'genre_cols'   : ALL_GENRES,
    'tag_cols'     : TAG_COLS,
    'feature_cols' : FEATURE_COLS,
    'user_mapping' : {str(k): v for k, v in user_mapping.items()},
    'item_mapping' : {str(k): v for k, v in item_mapping.items()},
}
with open(MODELS_DIR / 'fm_config.json', 'w') as f:
    json.dump(fm_config, f, indent=2)

item_embeddings = fm_model.get_item_embeddings(
    item_field_index=1,
    item_offset_within_field=range(n_items)
).numpy()
np.save(MODELS_DIR / 'item_embeddings.npy', item_embeddings)

print('Saved:')
print(f'  fm_model.pt          ({(MODELS_DIR/"fm_model.pt").stat().st_size/1024:.1f} KB)')
print(f'  fm_config.json       (genres={len(ALL_GENRES)}, tags={len(TAG_COLS)}, embed_dim=32)')
print(f'  item_embeddings.npy  shape={item_embeddings.shape}')

## 9. Sanity Check — sample predictions

In [ ]:
fm_model.eval()
sample = test_df.head(10).copy()
ds_sample = MovieLensDataset(sample, user_mapping, item_mapping, FEATURE_COLS)
xs = torch.stack([ds_sample[i][0] for i in range(len(ds_sample))])
with torch.no_grad():
    preds = fm_model(xs.to(DEVICE)).cpu().numpy()
sample['predicted'] = np.clip(preds, 0.5, 5.0)
sample[['userId', 'movieId', 'rating', 'predicted']]